# Procesamiento de Imágenes, audio y vídeo
## Práctica 5. Detección de características
### Ejercicio 1.  Desarrolle una aplicación que permita
**a)  A través de la interfaz modificar los parámetros del detector de características SIFT.**

**b) Seleccionar un área de interés en una imagen de elección una imagen de naturaleza médica y una imagen telemétrica.**

**c) Buscar esa área de interés (recuádrela en rojo) dentro de diferentes versiones de la imagen de partida (con cambios de traslación, escala y rotación) [NOTA: Estos cambios se pueden acometer con un editor de imágenes o con el trabajo hecho en prácticas previas]. Altere mediante la interfaz las configuraciones de parámetros para mejorar la detección.**

**d) Pruebe a hacer lo mismo que en el apartado c) con diferentes grados de deformación de la imagen. [NOTA: Estos cambios se pueden acometer con un editor de imágenes o con el trabajo hecho en prácticas previas].**

### Apartado A
Para este apartado, se ha desarrollado una interfaz gráfica utilizando OpenCV, que permita modificar los parámetros del detector de características SIFT. Los parámetros que se pueden ajustar son:
- nfeatures: Número máximo de características a detectar.
- nOctaveLayers: Número de capas por octava.
- contrastThreshold: Umbral de contraste para filtrar características débiles.
- edgeThreshold: Umbral para filtrar características en bordes.
- sigma: Desviación estándar de la función gaussiana utilizada para suavizar la imagen.


In [2]:
import cv2 as cv
import numpy as np
from pathlib import Path
import time

# ==============================
# Configuración inicial
# ==============================
IMG_PATH = "images/Cara_Diego.jpeg"  # Cambia esta ruta a tu imagen de prueba
win = "SIFT - Ajuste de parámetros"

# Intenta cargar imagen
img = cv.imread(IMG_PATH)
if img is None:
    raise FileNotFoundError(f"No se encontró la imagen en: {Path(IMG_PATH).resolve()}")

gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

# ==============================
# Utilidades
# ==============================
def nothing(_):  # callback vacío para trackbars
    pass

def map_range(val, in_min, in_max, out_min, out_max):
    # Mapea un entero de [in_min, in_max] a float en [out_min, out_max]
    return out_min + (out_max - out_min) * (val - in_min) / (in_max - in_min)

# ==============================
# Ventana + Trackbars
# ==============================
cv.namedWindow(win, cv.WINDOW_NORMAL)
cv.resizeWindow(win, 1280, 720)

# Rangos prácticos
# nfeatures: 0..10000
cv.createTrackbar("nfeatures", win, 1000, 10000, nothing)

# nOctaveLayers: 1..10
cv.createTrackbar("nOctaveLayers", win, 3, 10, nothing)

# contrastThreshold (float): 0.001..0.1 (escala log suave con slider lineal)
# Usamos slider 1..1000 y mapeamos a 0.001..0.1
cv.createTrackbar("contrast x1000", win, 50, 1000, nothing)  # 50 -> ~0.005

# edgeThreshold (float): 1..50 (OpenCV usa int/float; aquí lo tratamos como float)
cv.createTrackbar("edgeThr", win, 10, 100, nothing)

# sigma (float): 0.5..3.0
cv.createTrackbar("sigma x100", win, 150, 500, nothing)  # 150 -> 1.50

# ==============================
# Bucle principal
# ==============================
fps_deque = []
font = cv.FONT_HERSHEY_SIMPLEX

while True:
    t0 = time.time()

    # Leer valores de trackbars
    nfeatures      = cv.getTrackbarPos("nfeatures", win)
    nOctaveLayers  = max(1, cv.getTrackbarPos("nOctaveLayers", win))
    ct_raw         = cv.getTrackbarPos("contrast x1000", win)
    edge_thr_raw   = cv.getTrackbarPos("edgeThr", win)
    sigma_raw      = cv.getTrackbarPos("sigma x100", win)

    # Escalados a float
    contrastThreshold = map_range(max(1, ct_raw), 1, 1000, 0.001, 0.1)
    edgeThreshold     = map_range(max(1, edge_thr_raw), 1, 100, 1.0, 50.0)
    sigma             = map_range(max(1, sigma_raw), 1, 500, 0.5, 3.0)

    # Crear detector SIFT con parámetros actuales
    # (OpenCV >= 4.x)
    try:
        sift = cv.SIFT_create(
            nfeatures=int(nfeatures),
            nOctaveLayers=int(nOctaveLayers),
            contrastThreshold=float(contrastThreshold),
            edgeThreshold=float(edgeThreshold),
            sigma=float(sigma)
        )
    except AttributeError:
        raise RuntimeError("Tu OpenCV no tiene SIFT. Instala opencv-contrib-python o usa OpenCV>=4.4.")

    # Detectar y dibujar keypoints
    kps, desc = sift.detectAndCompute(gray, None)
    vis = cv.drawKeypoints(
        img, kps, None,
        flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
    )

    # Medir FPS suavizado
    dt = time.time() - t0
    fps = 1.0 / dt if dt > 0 else 0.0
    fps_deque.append(fps)
    if len(fps_deque) > 20:
        fps_deque.pop(0)
    fps_smoothed = sum(fps_deque) / len(fps_deque)

    # Overlay con datos
    overlay_lines = [
        f"keypoints: {len(kps)}",
        f"nfeatures={nfeatures}",
        f"nOctaveLayers={nOctaveLayers}",
        f"contrastThreshold={contrastThreshold:.4f}",
        f"edgeThreshold={edgeThreshold:.2f}",
        f"sigma={sigma:.2f}",
        f"FPS ~ {fps_smoothed:.1f} | (s: guardar, q/Esc: salir)"
    ]
    y = 24
    for line in overlay_lines:
        cv.putText(vis, line, (12, y), font, 0.7, (0, 0, 0), 3, cv.LINE_AA)
        cv.putText(vis, line, (12, y), font, 0.7, (255, 255, 255), 1, cv.LINE_AA)
        y += 28

    cv.imshow(win, vis)
    key = cv.waitKey(1) & 0xFF

    if key in (27, ord('q')):  # Esc o q
        break
    elif key == ord('s'):
        out_name = f"sift_preview_k{len(kps)}_nf{nfeatures}_nol{nOctaveLayers}_ct{contrastThreshold:.4f}_et{edgeThreshold:.2f}_sg{sigma:.2f}.png"
        cv.imwrite(out_name, vis)
        print(f"[Guardado] {out_name}")

cv.destroyAllWindows()


### Apartado B
Seleccionar un área de interés en una imagen de elección una imagen de naturaleza médica y una imagen telemétrica

In [11]:
import cv2 as cv
from pathlib import Path

IMG_PATH = "images/Cara_Diego.jpeg"  # cambia a imagen médica o telemétrica según el caso
win = "Selección de área de interés"

img = cv.imread(IMG_PATH)
if img is None:
    raise FileNotFoundError(f"No se encontró {Path(IMG_PATH).resolve()}")

clone = img.copy()
roi = None
drawing = False
ix, iy = -1, -1

def mouse_callback(event, x, y, flags, param):
    global ix, iy, drawing, roi, img
    if event == cv.EVENT_LBUTTONDOWN:
        drawing = True
        ix, iy = x, y
    elif event == cv.EVENT_MOUSEMOVE and drawing:
        img = clone.copy()
        cv.rectangle(img, (ix, iy), (x, y), (0, 0, 255), 2)
    elif event == cv.EVENT_LBUTTONUP:
        drawing = False
        roi = (min(ix, x), min(iy, y), abs(x - ix), abs(y - iy))
        cv.rectangle(img, (roi[0], roi[1]), (roi[0] + roi[2], roi[1] + roi[3]), (0, 0, 255), 2)

cv.namedWindow(win)
cv.setMouseCallback(win, mouse_callback)

while True:
    cv.imshow(win, img)
    k = cv.waitKey(1) & 0xFF
    if k in (27, ord('q')):  # salir
        break
    elif k == ord('s') and roi:
        x, y, w, h = roi
        crop = clone[y:y+h, x:x+w]
        cv.imwrite("images/roi_guardada.png", crop)
        print("ROI guardada como roi_guardada.png")

cv.destroyAllWindows()


ROI guardada como roi_guardada.png


### Apartado C
Buscar esa área de interés (recuádrela en rojo) dentro de diferentes versiones de la imagen de partida (con cambios de traslación, escala y rotación) [NOTA: Estos cambios se pueden acometer con un editor de imágenes o con el trabajo hecho en prácticas previas]. Altere mediante la interfaz las configuraciones de parámetros para mejorar la detección.

In [12]:
import cv2 as cv
import numpy as np
from pathlib import Path

# ==== Configuración ====
img_base_path = "images/Cara_Diego.jpeg"       # imagen original
roi_path = "images/roi_guardada.png"                  # ROI del apartado B
img_test_path = "images/Sota_Diego.png" # imagen transformada (traslación/escala/rotación)

# ==== Cargar imágenes ====
img1 = cv.imread(roi_path, cv.IMREAD_GRAYSCALE)       # ROI
img2 = cv.imread(img_test_path, cv.IMREAD_GRAYSCALE)  # Imagen transformada

if img1 is None or img2 is None:
    raise FileNotFoundError("No se encontraron las imágenes necesarias.")

# ==== Crear detector SIFT ====
sift = cv.SIFT_create(nfeatures=1000, contrastThreshold=0.02, edgeThreshold=10, sigma=1.6)

# ==== Detectar y describir ====
kp1, des1 = sift.detectAndCompute(img1, None)
kp2, des2 = sift.detectAndCompute(img2, None)

# ==== Emparejar descriptores con FLANN ====
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)
flann = cv.FlannBasedMatcher(index_params, search_params)
matches = flann.knnMatch(des1, des2, k=2)

# ==== Filtrado según razón de Lowe ====
good = []
for m, n in matches:
    if m.distance < 0.7 * n.distance:
        good.append(m)

# ==== Mostrar resultados ====
MIN_MATCH_COUNT = 10
img_matches = cv.drawMatches(img1, kp1, img2, kp2, good, None, flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

if len(good) > MIN_MATCH_COUNT:
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 5.0)
    h, w = img1.shape
    pts = np.float32([[0,0],[0,h-1],[w-1,h-1],[w-1,0]]).reshape(-1,1,2)
    dst = cv.perspectiveTransform(pts, M)
    img_result = cv.polylines(cv.cvtColor(img2, cv.COLOR_GRAY2BGR), [np.int32(dst)], True, (0,0,255), 3, cv.LINE_AA)
    cv.imshow("Resultado - ROI detectada", img_result)
else:
    print("No se encontraron suficientes coincidencias para localizar la ROI.")
    img_result = cv.cvtColor(img2, cv.COLOR_GRAY2BGR)
    cv.imshow("Resultado - No detectada", img_result)

cv.imshow("Coincidencias", img_matches)
cv.waitKey(0)
cv.destroyAllWindows()


No se encontraron suficientes coincidencias para localizar la ROI.


### Apartado D
Pruebe a hacer lo mismo que en el apartado c) con diferentes grados de deformación de la imagen. [NOTA: Estos cambios se pueden acometer con un editor de imágenes o con el trabajo hecho en prácticas previas].

In [13]:
import cv2 as cv
import numpy as np
from pathlib import Path

# ==== Configuración ====
roi_path = "images/roi_guardada.png"          # ROI del apartado B
base_path = "images/Cara_Diego.jpeg"   # Imagen original

# Lista de imágenes transformadas (pueden venir de tu editor)
test_images = [
    "images/Cara_Diego_rotada.jpg",
    "images/Cara_Diego_escalada.jpg",
    "images/Cara_Diego_trasladada.jpg",
    "images/Cara_Diego_deformada.jpg"
]

# ==== Cargar ROI ====
roi = cv.imread(roi_path, cv.IMREAD_GRAYSCALE)
if roi is None:
    raise FileNotFoundError("No se encontró la ROI guardada.")

# ==== Detector SIFT configurable ====
sift = cv.SIFT_create(nfeatures=1000, contrastThreshold=0.02, edgeThreshold=10, sigma=1.6)
kp_roi, des_roi = sift.detectAndCompute(roi, None)

# ==== Parámetros FLANN ====
index_params = dict(algorithm=1, trees=5)
search_params = dict(checks=50)
flann = cv.FlannBasedMatcher(index_params, search_params)

# ==== Procesar cada imagen de prueba ====
for img_path in test_images:
    img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
    if img is None:
        print(f"No se encontró {img_path}, se omite.")
        continue

    kp_img, des_img = sift.detectAndCompute(img, None)
    if des_img is None or des_roi is None:
        print(f"No se pudieron calcular descriptores para {img_path}")
        continue

    matches = flann.knnMatch(des_roi, des_img, k=2)
    good = [m for m, n in matches if m.distance < 0.7 * n.distance]

    img_color = cv.cvtColor(img, cv.COLOR_GRAY2BGR)

    if len(good) >= 10:
        src_pts = np.float32([kp_roi[m.queryIdx].pt for m in good]).reshape(-1,1,2)
        dst_pts = np.float32([kp_img[m.trainIdx].pt for m in good]).reshape(-1,1,2)
        M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 5.0)
        if M is not None:
            h, w = roi.shape
            pts = np.float32([[0,0],[0,h-1],[w-1,h-1],[w-1,0]]).reshape(-1,1,2)
            dst = cv.perspectiveTransform(pts, M)
            cv.polylines(img_color, [np.int32(dst)], True, (0,0,255), 3, cv.LINE_AA)
            print(f"ROI detectada en {img_path} con {len(good)} coincidencias.")
        else:
            print(f"Coincidencias insuficientes en {img_path}.")
    else:
        print(f"No se encontró la ROI en {img_path} ({len(good)} coincidencias).")

    cv.imshow(f"Detección en {Path(img_path).stem}", img_color)
    cv.waitKey(0)

cv.destroyAllWindows()


ROI detectada en images/Cara_Diego_rotada.jpg con 303 coincidencias.
ROI detectada en images/Cara_Diego_escalada.jpg con 306 coincidencias.
ROI detectada en images/Cara_Diego_trasladada.jpg con 297 coincidencias.
ROI detectada en images/Cara_Diego_deformada.jpg con 127 coincidencias.
